In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

print("✅ Imports successful!")
print(f"   PyTorch version: {torch.__version__}")
print(f"   CUDA available: {torch.cuda.is_available()}")

✅ Imports successful!
   PyTorch version: 2.11.0+cpu
   CUDA available: False


In [ ]:
MODEL_NAME = "microsoft/DialoGPT-medium"

print(f"⏳ Loading model: {MODEL_NAME}")
print("   This may take a minute on first run (downloads ~1.4 GB)...\n")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

model.eval()

print("\n✅ Model loaded successfully!")
print(f"   Model parameters: {sum(p.numel() for p in model.parameters()):,}")

⏳ Loading model: microsoft/DialoGPT-medium
   This may take a minute on first run (downloads ~1.4 GB)...



config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

d:\Anaconda\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\omdon\.cache\huggingface\hub\models--microsoft--DialoGPT-medium. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/863M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]


✅ Model loaded successfully!
   Model parameters: 354,823,168


model.safetensors:   0%|          | 0.00/863M [00:00<?, ?B/s]

In [ ]:
def generate_response(user_input, conversation_history, max_history_turns=5):

    new_input_ids = tokenizer.encode(
        user_input + tokenizer.eos_token,  
        return_tensors='pt'                
    )
    
    if len(conversation_history) > max_history_turns:
        conversation_history = conversation_history[-max_history_turns:]
    
    if conversation_history:
        bot_input_ids = torch.cat([*conversation_history, new_input_ids], dim=-1)
    else:
        bot_input_ids = new_input_ids  
    

    with torch.no_grad():  
        output_ids = model.generate(
            bot_input_ids,
            max_length=1000,        
            pad_token_id=tokenizer.eos_token_id,  
            do_sample=True,        
            temperature=0.75,       
            top_p=0.9,              
            top_k=50,               
            repetition_penalty=1.3  
        )
    
 
    response_ids = output_ids[:, bot_input_ids.shape[-1]:]
    response_text = tokenizer.decode(
        response_ids[0],
        skip_special_tokens=True  # Remove <|endoftext|> tokens from output
    )
    
    # Update history with full output (for next turn's context)
    conversation_history.append(output_ids)
    
    return response_text.strip(), conversation_history


print("✅ Response generation function defined!")

✅ Response generation function defined!


In [4]:
# Quick test with a single message
print("🧪 Testing model with a sample input...\n")

test_history = []
test_response, test_history = generate_response("Hello! How are you?", test_history)

print(f"User : Hello! How are you?")
print(f"Bot  : {test_response}")
print("\n✅ Model is working correctly!")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


🧪 Testing model with a sample input...

User : Hello! How are you?
Bot  : I'm good!

✅ Model is working correctly!


In [ ]:
def run_chatbot():
    print("=" * 55)
    print("   Type 'exit' or 'quit' to end the session")
    print("=" * 55)
    print("\nChatbot: Hello! I am your AI assistant. How can I help you today?\n")
    
    conversation_history = []  # Stores token tensors for multi-turn context
    turn_count = 0
    
    while True:
        # Accept user input
        try:
            user_input = input("You: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nChatbot: Session interrupted. Goodbye! 👋")
            break
        
        # Skip empty inputs
        if not user_input:
            print("Chatbot: Please type something! I'm listening...\n")
            continue
        
        # Exit condition
        if user_input.lower() in ["exit", "quit"]:
            print("\nChatbot: Thank you for chatting with me! Have a great day! 👋")
            print(f"         Total conversation turns: {turn_count}")
            print("=" * 55)
            break
        
        # Generate response using transformer model
        print("Chatbot: ", end="", flush=True)  # Show prompt while generating
        
        response, conversation_history = generate_response(
            user_input, conversation_history
        )
        
        # Handle edge case: empty/very short response from model
        if not response or len(response) < 3:
            response = "That's interesting! Could you tell me more about that?"
        
        print(f"{response}\n")
        turn_count += 1

run_chatbot()

   🤖 DialoGPT Conversational Chatbot
   Powered by: microsoft/DialoGPT-medium
   Type 'exit' or 'quit' to end the session

Chatbot: Hello! I am your AI assistant. How can I help you today?

Chatbot: Good to see you again, u Pizzapig! Hope we can get some more of us in here.

Chatbot: A programming language that uses symbols and a lot of things from the Matrix universe, I believe :D

Chatbot: Please type something! I'm listening...


Chatbot: Thank you for chatting with me! Have a great day! 👋
         Total conversation turns: 2
